---

## Step-by-Step Explanation of Multinomial Logit (MNL) Model

1. **Dataset Preparation**
   - Start with a dataset containing observations of individuals/entities making choices among multiple alternatives, including characteristics of the alternatives and the chosen alternative.

2. **Utility Function**
   - Assume a utility function representing the preferences of individuals for each alternative.
   - Utility is a combination of alternative-specific variables (ASV) and generic variables (GV).
   - ASVs capture characteristics specific to each alternative (e.g., cost, time, availability).
   - GVs represent characteristics common across all alternatives (e.g., age, gender, income).

3. **Utility Equation**
   - Express the utility of each alternative using a linear equation.
   - The utility equation for alternative j is:
     $U_{ij} = \beta_{0j} + \beta_{1j}X_{1ij} + \beta_{2j}X_{2ij} + \ldots + \beta_{kj}X_{kij}$
     where i represents the individual, j represents the alternative, and $X_{ij}$ represents the values of the ASVs and GVs for individual i and alternative j.
   - The $\beta_{kj}$ coefficients represent the relative importance of each variable in determining the utility of alternative j.

4. **Choice Probability**
   - Calculate the choice probability for each alternative using the utility equation and a reference alternative (pivot).
   - Choose a reference alternative (e.g., alternative A) as the baseline for comparison.
   - The utility of the reference alternative is set to zero: U_{iA} = 0 for all individuals i.
   - The utility equation for alternative j (j ≠ A) is modified to be relative to the reference alternative:
     $U_{ij} - U_{iA} = \beta_{0j} + \beta_{1j}X_{1ij} + \beta_{2j}X_{2ij} + \ldots + \beta_{kj}X_{kij}$
   - The choice probability of alternative j for individual i is given by the softmax function:
     $P_{ij} = \frac{e^{(U_{ij} - U_{iA})}}{\sum_{m=1}^{M}e^{(U_{im} - U_{iA})}}$
     where M is the total number of alternatives.

5. **Log-Likelihood Function**
   - Construct the log-likelihood function to estimate the model parameters.
   - The log-likelihood function is:
     $LL(\beta) = \sum_{i=1}^{N} \sum_{j=1}^{M} \left( y_{ij} \cdot \ln(P_{ij}) \right)$
     where N is the total number of observations and $y_{ij}$ is an indicator variable that equals 1 if individual i chooses alternative j, and 0 otherwise.

6. **Maximum Likelihood Estimation (MLE)**
   - Estimate the model parameters by maximizing the log-likelihood function.
   - The MLE method finds the values of the $\beta_{kj}$ coefficients that maximize the likelihood of observing the choices made in the dataset.
   - The optimization process adjusts the $\beta_{kj}$ coefficients iteratively to find the best fit for the observed choices.

7. **Interpretation of Coefficients**
   - After estimating the model, interpret the coefficients to understand the impact of variables on the choice probabilities.
   - Since the reference alternative (e.g., alternative A) was used as the baseline, the estimated coefficients represent the differences in utility between each alternative and the reference alternative.
   - Positive coefficients indicate that an increase in the variable value increases the utility compared to the reference alternative, while negative coefficients indicate the opposite effect.
   - The magnitude of the coefficients reflects the strength of the influence of the corresponding variable on the choice probabilities.

8. **Predictions**
   - Use the estimated model to make predictions for new observations.
   - Calculate the choice probabilities for each alternative (relative to the reference alternative) and choose the alternative with the highest probability as the predicted choice.

The concept of a pivot or reference alternative allows us to compare the utilities and choice probabilities of different alternatives relative to a baseline. By setting the utility of the reference alternative to zero and expressing the utility equations and choice probabilities in relation to the reference alternative, we can estimate the differences in utility and understand the relative preferences for each alternative.

Note: In practice, the choice of the reference alternative is arbitrary and should be chosen based on domain knowledge or specific research objectives.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('data_incentive_attitudes.csv')
df.head()

,Unnamed: 0,user_id,trip_id,departure_time,arrival_time,Purpose,buscost,bustime,carcost,cartime,...,SEX,AGE,INCOME,recco,information,incentivezone,value,task,job_type,income_con
0,1,20006,99545,10/28/2020 17:51,10/28/2020 19:21,101,770,69.4,151.7,43.72,...,1,36,4,2,only enviro,0,1,1,company employee,9000000
1,2,20006,100663,10/29/2020 17:58,10/29/2020 19:14,101,690,66.2,128.6,38.96,...,1,36,4,2,only enviro,0,1,2,company employee,9000000
2,3,20006,101712,10/30/2020 17:39,10/30/2020 19:05,101,770,69.4,151.7,43.72,...,1,36,4,1,no info,0,1,3,company employee,9000000
3,4,20006,101878,10/30/2020 21:00,10/30/2020 21:25,998,500,56.2,118.5,29.53,...,1,36,4,1,no info,0,1,4,company employee,9000000
4,5,20006,100936,10/30/2020 7:14,10/30/2020 8:29,100,770,69.4,151.7,43.72,...,1,36,4,1,no info,0,1,5,company employee,9000000


In [3]:
df = df.drop('Unnamed: 0', axis='columns')

In [4]:
df = df.rename(columns={'pubcost':'traincost', 
                   'pubtime':'traintime', 
                   'pubavail':'trainavail',
                    'bicycleincentive':'bikeincentive'})

In [5]:
# What is `recco`?

In [6]:
# Exogenous variables
basic_variables = list(df.columns)[:5]

purpose_code_dict = {100: 'Commuting to work / school',
                    101: 'Go Home',
                    200: 'Shopping for daily necessities',
                    201: 'Shopping other than daily necessities',
                    202: 'Meals and entertainment',
                    300: 'business',
                    400: 'Outpatient',
                    500: 'Pick-up and drop-off',
                    600: 'Sightseeing / Leisure',
                    998: 'others'}

alternative_specific_variables = list(df.columns)[5:29]
alternative_specific_variables.remove('mode')

attitudinal_variables = list(df.columns)[29:-10]
generic_variables = ['SEX', 'AGE', 'INCOME', 'job_type', 'information']

# <ins><b>Data Preprocessing</b></ins>

## Feature Encoding

In [8]:
from sklearn.preprocessing import OneHotEncoder
from datetime import datetime

In [9]:
ohe = OneHotEncoder()

In [10]:
# one-hot encoding the job_type column
#print(df['job_type'].value_counts())
job_type_sparse_matrix = ohe.fit_transform(df['job_type'].to_numpy().reshape(-1, 1)).toarray()
job_type_column_names = ['job_type_'+job for job in df['job_type'].unique()]

job_type_df = pd.DataFrame(data=job_type_sparse_matrix, columns=job_type_column_names)
#job_type_df.head()

In [12]:
# one-hot encoding the `information` column
#print(df['information'].value_counts())
info_sparse_matrix = ohe.fit_transform(df['information'].to_numpy().reshape(-1, 1)).toarray()
info_df = pd.DataFrame(data=info_sparse_matrix, columns=ohe.categories_[0])
#info_df.head()

In [13]:
# one-hot encoding the SEX column
#print(df['SEX'].value_counts())
SEX_sparse_matrix = ohe.fit_transform(df['SEX'].to_numpy().reshape(-1, 1)).toarray()
SEX_df = pd.DataFrame(data=SEX_sparse_matrix, columns=['Male', 'Female'])
#SEX_df.head()

In [14]:
# one-hot encoding the purpose column
#print(df['Purpose'].value_counts())

Purpose_sparse_matrix = ohe.fit_transform(df['Purpose'].to_numpy().reshape(-1, 1)).toarray()
Purpose_column_names = ['Purpose_'+str(purpose_code) for purpose_code in df['Purpose'].unique()]
Purpose_df = pd.DataFrame(data=Purpose_sparse_matrix, columns=Purpose_column_names)
#Purpose_df.head()

In [17]:
def get_trip_duration(df):
    """
    Returns the total minutes a trip lasted given its departure and arrival datetimes.
    """
    dep_datetime = datetime.strptime(df['departure_time'], '%m/%d/%Y %H:%M')
    arr_datetime = datetime.strptime(df['arrival_time'], '%m/%d/%Y %H:%M')
    
    return int((arr_datetime - dep_datetime).total_seconds()/60)

def get_day_zones(trip_datetime):
    """
    Divides the time into 4 zones and ordinally encoding them.
    Early morning (00:00 to 6:00) - 1
    AM peak (6:00 to 10:00) - 2
    Off peak (10:00 to 16:00) - 3
    PM peak (16:00 to 20:00) - 4
    Evening (20:00 to 00:00) - 5
    """
    dep_hrs, dep_mins = [int(val) for val in trip_datetime.split(' ')[1].split(':')]
    if dep_hrs >= 0 and dep_hrs < 6:
        return 1
    elif dep_hrs >= 6 and dep_hrs < 10:
        return 2
    elif dep_hrs >= 10 and dep_hrs < 16:
        return 3
    elif dep_hrs >= 16 and dep_hrs < 20:
        return 4
    else:
        return 5

In [20]:
# Getting the trip duration in minutes
df['trip_duration'] = df[['departure_time', 'arrival_time']].T.apply(get_trip_duration)

# Converting departure time in 4 zones and one-hot encoding it
dep_time_zones_sparse_matrix = ohe.fit_transform(df['departure_time'].apply(get_day_zones).to_numpy().reshape(-1, 1)).toarray()
dep_time_zones_names = ['Early_morning_departure', 'AM_peak_departure', 'Off_peak_departure', 'PM_peak_departure', 'Night_departure']
dep_time_zones_df = pd.DataFrame(data=dep_time_zones_sparse_matrix, columns=dep_time_zones_names)
#dep_time_zones_df

In [21]:
# combining all the one-hot encoded dataframes with the main dataframe
df2 = pd.concat([df, dep_time_zones_df, Purpose_df, SEX_df, job_type_df, info_df], axis=1).drop(
    ['departure_time', 'arrival_time', 'Purpose', 'SEX', 'job_type'], axis='columns')
df2.head()

,user_id,trip_id,buscost,bustime,carcost,cartime,cardistance,traincost,traintime,walktime,...,job_type_Management executive,job_type_civil servant,job_type_Housewife,job_type_Self employed/ Freelance,job_type_Others,job_type_Unemployed,both enviro and health,no info,only enviro,only health
0,20006,99545,770,69.4,151.7,43.72,15.17,240,60.9,182.04,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,20006,100663,690,66.2,128.6,38.96,12.86,240,44.9,154.32,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,20006,101712,770,69.4,151.7,43.72,15.17,240,60.9,182.04,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
3,20006,101878,500,56.2,118.5,29.53,11.85,400,59.0,142.20,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,20006,100936,770,69.4,151.7,43.72,15.17,240,60.9,182.04,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


In [22]:
# Removing some unnecessary columns
df2 = df2.drop(['user_id', 'trip_id', 'recco', 'information', 'incentivezone', 'value', 'task', 'income_con'], axis='columns')

## Dealing with outliers

In [23]:
df3 = df2.copy()

In [24]:
# Quantile based winsorization on trip duration
Q1 = df2['trip_duration'].quantile(0.25)
Q3 = df2['trip_duration'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 2*IQR
trip_durations = df2['trip_duration'].to_numpy()
trip_durations = np.where((trip_durations <= upper_bound), trip_durations, upper_bound)

# fig, ax = plt.subplots(1, 2, figsize=(12, 5))
# sns.histplot(trip_durations, ax=ax[1])
# sns.histplot(df2['trip_duration'], ax=ax[0])
# ax[0].set_title('Pre processing')
# ax[1].set_title('Post processing')
# ax[1].set_xlabel('Trip Duration'); ax[0].set_xlabel('Trip Duration')

df3['trip_duration'] = trip_durations

In [25]:
# fig, ax = plt.subplots(len(alternative_specific_variables[:-6]), 2, 
#                        figsize=(12, 4*len(alternative_specific_variables[:-6])), squeeze=False, gridspec_kw={'hspace':0.5})

for i in range(len(alternative_specific_variables[:-6])):
    Q1 = df2[alternative_specific_variables[i]].quantile(0.25)
    Q3 = df2[alternative_specific_variables[i]].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.9*IQR
    lower_bound = Q1 - 1.9*IQR
    asc_variable = df2[alternative_specific_variables[i]].to_numpy()
    df3[alternative_specific_variables[i]] = np.where((asc_variable <= upper_bound) & (asc_variable >= lower_bound), 
                                                      asc_variable, upper_bound)

#     _ = ax[i, 1].hist(df3[alternative_specific_variables[i]], bins=20)
#     _ = ax[i, 0].hist(df2[alternative_specific_variables[i]], bins=20)
#     ax[i, 0].set_title('Pre processing')
#     ax[i, 1].set_title('Post processing')
#     ax[i, 1].set_xlabel(alternative_specific_variables[i])
#     ax[i, 0].set_xlabel(alternative_specific_variables[i])

In [26]:
# Betting rid of bikecost and walkcost (no variability)
df3 = df3.drop(['bikecost', 'walkcost'], axis='columns')

## Train and Test Split

In [86]:
from sklearn.model_selection import train_test_split

In [87]:
train_df, test_df = train_test_split(df3, test_size=0.2)
train_df.shape, test_df.shape

((2468, 123), (618, 123))

In [88]:
train_df = train_df.drop(attitudinal_variables, axis='columns')
test_df = test_df.drop(attitudinal_variables, axis='columns')

In [89]:
selected_mode = train_df['mode'].to_numpy()
train_df['mode'] = np.where(selected_mode=='pub', 'train', selected_mode)
selected_mode = test_df['mode'].to_numpy()
test_df['mode'] = np.where(selected_mode=='pub', 'train', selected_mode)

In [95]:
y_train = pd.get_dummies(train_df['mode'])
X_train = train_df.drop('mode', axis='columns')

y_test = pd.get_dummies(test_df['mode'])
X_test = test_df.drop('mode', axis='columns')

In [96]:
output_categories = list(train_df['mode'].unique())
generic_variables = list(train_df.columns)[22:]
asc_variable_names = {
    'bus': ['buscost', 'bustime', 'busincentive'],
    'car': ['carcost', 'cartime', 'carincentive'],
    'train': ['traincost', 'traintime', 'trainincentive'], 
    'walk': ['walktime', 'walkincentive'],
    'bike': ['biketime', 'bikeincentive'],
    'motor': ['motorcost', 'motortime', 'motorincentive']
}

asc_variable_indexer = dict()
for category in output_categories:
    asc_variable_indexer[category] = X_train.columns.get_indexer(
                                    asc_variable_names[category])

# <ins><b>Model Formulation</b></ins>

In [97]:
import tensorflow as tf

In [98]:
class AltSpecDense(tf.keras.layers.Layer):
    def __init__(self, categories=output_categories, asc_variables=asc_variable_indexer):
        super(AltSpecDense, self).__init__()
        self.units=len(categories)
        self.categories = categories
        self.asc_variables = asc_variables
        
    def build(self, input_shape):
        w_init = tf.random_normal_initializer()
        self.w = []
        for category in self.categories:
            n = len(self.asc_variables[category])
            self.w.append(tf.Variable(
                            initial_value=w_init(shape=(n, 1), 
                                                 dtype='float32'),
                            trainable=True, name=f'{category}_weights'))
        
        # Initializing the bias tensor
        b_init = tf.zeros_initializer()
        self.b = tf.Variable(
                initial_value=b_init(shape=(self.units,), dtype='float32'),
                trainable=True, name='biases')
    
    
    def call(self, inputs):
        total_alt_spec_utility = []
        # Calulating the utility of each category (predicting class) without the asc constant (bias)
        for i, category in enumerate(self.categories):
            indices = tf.constant(self.asc_variables[category])
            sliced_inputs = tf.gather(inputs, indices, axis=1)
            curr_utility = tf.matmul(sliced_inputs, self.w[i])
            total_alt_spec_utility.append(curr_utility)
        total_alt_spec_utility = tf.concat(total_alt_spec_utility, axis=1)
        return total_alt_spec_utility + self.b

In [100]:
X_train.columns[:21]

Index(['buscost', 'bustime', 'carcost', 'cartime', 'cardistance', 'traincost',
       'traintime', 'walktime', 'biketime', 'motortime', 'motorcost',
       'carincentive', 'busincentive', 'walkincentive', 'trainincentive',
       'bikeincentive', 'motorincentive', 'busavail', 'trainavail',
       'bikeavail', 'motoravail'],
      dtype='object')

In [37]:
class MNLModel(tf.keras.models.Model):
    def __init__(self):
        super(MNLModel, self).__init__()
        self.generic_utility_layer = tf.keras.layers.Dense(6, use_bias=False)
        self.alt_spec_utility_layer = AltSpecDense()
    
    def call(self, x):
        generic_x = x[:, 21:]
        alt_spec_x = x[:, :21]
        
        generic_utility = self.generic_utility_layer(generic_x)
        alt_spec_utility = self.alt_spec_utility_layer(alt_spec_x)
        
        return generic_utility + alt_spec_utility

In [38]:
def replace_nan_with_zeros(tensor):
    nan_mask = tf.math.is_nan(tensor)
    zeros_mask = tf.zeros_like(tensor)
    replaced_tensor = tf.where(nan_mask, zeros_mask, tensor)
    return replaced_tensor

In [110]:
X_train.to_numpy()[:, 17:21]

array([[1., 1., 0., 0.],
       [1., 1., 0., 0.],
       [0., 0., 1., 0.],
       ...,
       [1., 1., 0., 0.],
       [0., 0., 0., 0.],
       [1., 1., 1., 0.]])

In [120]:
y_train

,bike,bus,car,motor,train,walk
2722,0,0,1,0,0,0
3024,0,0,1,0,0,0
210,0,0,1,0,0,0
1872,0,0,1,0,0,0
209,0,0,1,0,0,0
...,...,...,...,...,...,...
7,0,0,1,0,0,0
650,0,0,1,0,0,0
1027,0,0,1,0,0,0
940,0,0,1,0,0,0


In [184]:
class SparseCategoricalCrossEntropy(tf.keras.losses.Loss):
    
    def __init__(self, from_logits=True, reduction=tf.keras.losses.Reduction.AUTO, name='sparse_categorical_crossentropy'):
        super(SparseCategoricalCrossEntropy, self).__init__(reduction=reduction, name=name)
        self.from_logits = from_logits
        
    def call(self, true_y, pred_y):
        if self.from_logits:
            pred_y = tf.nn.softmax(pred_y, axis=-1)
        
        batch_avail_data = self.avail_data
        
        # Masking the original softmax probabilities based on availability data
        # and normalizing the masked probabilities to effectively negate the effect 
        # of non-available label class data.
        masked_probs = pred_y * tf.cast(batch_avail_data, dtype=pred_y.dtype)
        summed_probs = tf.reduce_sum(masked_probs, axis=-1, keepdims=True)
        pred_y = masked_probs / (summed_probs + tf.keras.backend.epsilon())      
        
        loss = -tf.reduce_sum(replace_nan_with_zeros(true_y * tf.math.log(pred_y + tf.keras.backend.epsilon())))
        self.avail_data = None
        return loss
    
    def set_avail_data(self, avail_data):
        self.avail_data = avail_data

In [185]:
mnl_model = MNLModel()

In [186]:
loss_object = SparseCategoricalCrossEntropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()

In [187]:
train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

test_loss = tf.keras.metrics.Mean(name='test_loss')
test_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='test_accuracy')

In [138]:
np.hstack((y_train.to_numpy(), X_train.to_numpy()[:, 17:21]))[:, -4:]

array([[1., 1., 0., 0.],
       [1., 1., 0., 0.],
       [0., 0., 1., 0.],
       ...,
       [1., 1., 0., 0.],
       [0., 0., 0., 0.],
       [1., 1., 1., 0.]])

In [188]:
@tf.function
def train_step(train_X, train_y, avail_data):
    # Loading all the performed operations while predicting and calculating loss
    with tf.GradientTape() as tape:
        predictions = mnl_model(train_X)
        loss_object.set_avail_data(avail_data)
        curr_train_loss = loss_object(train_y, predictions)
    
    # Calculating gradients using automatic differentiation based on loaded operations
    gradients = tape.gradient(curr_train_loss, mnl_model.trainable_variables)
    
    # Updating the trainable variables using the computed gradients.
    optimizer.apply_gradients(zip(gradients, mnl_model.trainable_variables))
    
    # Logging the loss and accuracy at each training step for the training batch
    train_loss(curr_train_loss)
    train_accuracy(train_y, predictions)


In [ ]:
tf.

In [189]:
@tf.function
def test_step(test_X, test_y, avail_data):
    predictions = model(test_X.to_numpy())
    loss_object.set_avail_data(avail_data)
    curr_test_loss = loss_object(test_y, predictions)
    
    # Logging the loss and accuracy at each testing step for the testing batch
    test_loss_obj(curr_test_loss)
    test_accuracy_obj(test_y, predictions)

In [190]:
BATCH_SIZE = 20

train_ds = tf.data.Dataset.from_tensor_slices(
        (X_train.to_numpy(), y_train.to_numpy())).shuffle(1000).batch(BATCH_SIZE)
test_ds = tf.data.Dataset.from_tensor_slices(
        (X_test.to_numpy(), y_test.to_numpy())).batch(BATCH_SIZE)

In [ ]:
def create_avail_data(X, label_order=list(y_train.columns)):
    
    

In [193]:
list(y_train.columns)

['bike', 'bus', 'car', 'motor', 'train', 'walk']

In [191]:
EPOCHS = 5

for epoch in range(EPOCHS):
    # Reset the metrics at the start of the next epoch
    train_loss.reset_states()
    train_accuracy.reset_states()
    test_loss.reset_states()
    test_accuracy.reset_states()

    for xtrain, ytrain in train_ds:
        train_step(xtrain, ytrain, xtrain[:, 17:21].numpy())

    for xtest, ytest in test_ds:
        test_step(xtest, ytest, xtest[:, 17:21].numpy())

    print(
    f'Epoch {epoch + 1}, '
    f'Loss: {train_loss.result()}, '
    f'Accuracy: {train_accuracy.result() * 100}, '
    f'Test Loss: {test_loss.result()}, '
    f'Test Accuracy: {test_accuracy.result() * 100}'
    )

ValueError: in user code:

    File "<ipython-input-171-d3064a4f007d>", line 7, in train_step  *
        curr_train_loss = loss_object(train_y, predictions)
    File "<ipython-input-184-5acfe5b1479b>", line 16, in call  *
        masked_probs = pred_y * tf.cast(batch_avail_data, dtype=pred_y.dtype)

    ValueError: Dimensions must be equal, but are 6 and 4 for '{{node sparse_categorical_crossentropy/mul}} = Mul[T=DT_FLOAT](sparse_categorical_crossentropy/Softmax, sparse_categorical_crossentropy/Cast)' with input shapes: [20,6], [20,4].


In [151]:
total_trainable_variables=0
for var in mml.trainable_variables:
    total_trainable_variables+=var.shape[0]
total_trainable_variables

NameError: name 'mml' is not defined